## Working directory and file paths

When started from the project root, the first code cell switches the working directory to shared_python. If it is already there, it leaves it unchanged. Input arrays are read from ../Data/. Generated data and figures are saved to ../data/ and ../figures/; the setup cell creates those output folders.


In [ ]:
import os
import sys

if os.path.isdir("shared_python"):
    os.chdir("shared_python")

os.makedirs("../data", exist_ok=True)
os.makedirs("../figures", exist_ok=True)
sys.path.insert(0, os.path.abspath("../Code_main"))


## Model setup and helper functions

This section imports the Python libraries and compiled renewal-equation solvers, then defines the numerical helpers and distribution functions used by the model. The following cells configure the variant subsets, approximation methods, time grid, and baseline parameters.


In [3]:
import infectionRenewalEq_x86_v3
import infectionRenewalEq_seqcirc_approx
from scipy.signal import fftconvolve
from scipy.signal import convolve

import numpy as np
import scipy.integrate as spi
from scipy.integrate import cumulative_trapezoid
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
from itertools import chain, combinations
import time
from tqdm import tqdm
import pandas as pd

def powerset(iterable):
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def myconvolution_fft(g,f,hind):
    g = np.asarray(g)
    f = np.asarray(f)
    n = len(g)
    
    # Part 1: convolution (g with reversed f)
    dot_terms = fftconvolve(g, f, mode='full')[:n]
    
    # Part 2: suffix sum of g
    if hind==0:
        suffix_sum = np.cumsum(g[::-1])[::-1]
        dot_terms = dot_terms + suffix_sum
    
    return dot_terms
    

In [4]:
def init_sigma_nu_h(sigma_pair,nu_values,h_values,method='min'):

    sigma_nu_h = np.ones((len(nu_values), len(h_values)))
    
    for i, nu in enumerate(nu_values):
        for j, h in enumerate(h_values):
            if len(h)==1:
                sigma_nu_h[i, j] = sigma_pair[i,(j-1)]   
    
     # Fill higher-order h-values
    if method=='min':
        for i, nu in enumerate(nu_values):
            for j, h in enumerate(h_values):
                if len(h)>1:
                    sigma_nu_h[i, j] = np.array([sigma_nu_h[i, ii] for ii in h]).min()
    if method=='prod':
        for i, nu in enumerate(nu_values):
            for j, h in enumerate(h_values):
                if len(h)>1:
                    sigma_nu_h[i, j] = np.prod(np.array([sigma_nu_h[i, ii] for ii in h]))
 
    # Set entries to zero if nu in h   
    for i, nu in enumerate(nu_values):
        for j, h in enumerate(h_values):
            if nu in h:
                sigma_nu_h[i, j] = 0
    return sigma_nu_h


In [5]:
def mean_sd_to_alpha_scale_gamma_dist(mean, sd):
    var = sd**2
    alpha = (mean ** 2) / var
    scale = var/mean
    return alpha, scale
    
## Eponantial distribution for generation time and recovery time:
# gamma_u_h = lambda tau, nu, h: gamma * np.exp(-gamma * tau) * (tau >= 0)
# g_nu_h = lambda tau, nu, h: stats.gamma.pdf(tau, a=alpha, scale=scale) * (tau >= 0)  

## Gamma distribution for generation time and recovery time:
gamma_u_h = lambda tau, alpha, scale, nu, h: stats.gamma.pdf(tau , a=alpha, scale=scale) * (tau >= 0)
g_nu_h = lambda tau, alpha, scale, nu, h: stats.gamma.pdf(tau , a=alpha, scale=scale) * (tau >= 0)

def l_nu_h(t_values, alpha, scale, nu, h):
    integral_values = spi.cumulative_simpson(y=gamma_u_h(t_values,alpha, scale, nu, h), x=t_values, initial=0)
    # integral_values = spi.cumulative_trapezoid(y=gamma_u_h(t_values, nu, h), x=t_values, initial=0)    
    # integral_values = spi.quad(lambda tau: gamma_u_h(tau, alpha, scale, nu, h), 0, t_values)[0]
    return (1-integral_values)

def l_nu_h2(tau, alpha, scale, nu, h):
    integral_values = stats.gamma.cdf(tau , a=alpha, scale=scale)
    return (1-integral_values) * (tau >= 0)
    
I_seed = lambda tau, alpha, scale, nu, h: np.zeros(len(tau)) if h else  I0*l_nu_h2(tau, alpha, scale, nu, h) * (tau >= 0)
IntgImptG1_t =  lambda tau, alpha, scale, nu, h, I0: np.zeros(len(tau)) if h else  I0*g_nu_h(tau, alpha, scale, nu, h)
def IntgImptG2_t(t_values, alpha, scale, nu, h, I0):
    if h:
        return np.zeros(len(t_values))  # Return 0 if h is empty
    integral_values = spi.cumulative_simpson(y=g_nu_h(t_values, alpha, scale, nu, h), x=t_values, initial=0)
    # integral_values = spi.cumulative_trapezoid(g_nu_h(t_values, alpha, scale, nu, h), t_values, initial=0)
    # integral_values = spi.quad(lambda tau: g_nu_h(tau, nu, h), 0, t_values)[0]
    L = spi.quad(lambda tau: l_nu_h(tau, alpha, scale, nu, h), 0, 1000)[0]
    return (I0/L)*(1-integral_values)

IntgImptGamma1_t =  lambda tau, alpha, scale, nu, h, I0: np.zeros(len(tau)) if h else  I0*gamma_u_h(tau, alpha, scale, nu, h)
def IntgImptGamma2_t(t_values, alpha, scale, nu, h, I0):
    if h:
        return np.zeros(len(t_values))  # Return 0 if h is empty
    integral_values = spi.cumulative_simpson(y=gamma_u_h(t_values, alpha, scale, nu, h), x=t_values, initial=0)
    # integral_values = spi.cumulative_trapezoid(gamma_u_h(t_values, alpha, scale, nu, h), t_values, initial=0)
    # integral_values = spi.quad(lambda tau: gamma_u_h(tau, alpha, scale, nu, h), 0, t_values)[0]
    L = spi.quad(lambda tau: l_nu_h(tau, alpha, scale, nu, h), 0, 1000)[0]
    return (I0/L)*(1-integral_values)


In [6]:
# nu_values = [1]
# nu_values = [1, 2, 3, 4, 5]
nu_values = [1, 2, 3, 4]
nu_index_map = {nu: i for i, nu in enumerate(nu_values)}
h_values_all = list(map(set, powerset(nu_values))) 

nu_index_map = {nu: i for i, nu in enumerate(nu_values)} 
h_index_map = {tuple(h): i for i, h in enumerate(h_values_all)}
h_values_all_label = {tuple(h): str(h) for h in h_values_all}
h_values_all_label[()] = '\\emptyset'
print(h_values_all)
print(h_values_all_label)


[set(), {1}, {2}, {3}, {4}, {1, 2}, {1, 3}, {1, 4}, {2, 3}, {2, 4}, {3, 4}, {1, 2, 3}, {1, 2, 4}, {1, 3, 4}, {2, 3, 4}, {1, 2, 3, 4}]
{(): '\\emptyset', (1,): '{1}', (2,): '{2}', (3,): '{3}', (4,): '{4}', (1, 2): '{1, 2}', (1, 3): '{1, 3}', (1, 4): '{1, 4}', (2, 3): '{2, 3}', (2, 4): '{2, 4}', (3, 4): '{3, 4}', (1, 2, 3): '{1, 2, 3}', (1, 2, 4): '{1, 2, 4}', (1, 3, 4): '{1, 3, 4}', (2, 3, 4): '{2, 3, 4}', (1, 2, 3, 4): '{1, 2, 3, 4}'}


In [7]:
h_values_Jappx0 = [set()] 
# h_values_Jappx1 = [set(),{1},{2},{3}] 
h_values_Jappx1 = [set(), {1}, {2}, {3}, {4}]
# h_values_Jappx1 = [set(),{1}, {2}, {3}, {4}, {5}]
# h_values_Jappx2 = [set(),{1},{2},{3},{1, 2}, {1, 3}, {2, 3}] 
h_values_Jappx2 = [set(), {1}, {2}, {3}, {4}, {1, 2}, {1, 3}, {1, 4}, {2, 3}, {2, 4}, {3, 4}]
# h_values_Jappx2 =[set(), {1}, {2}, {3}, {4}, {5},{1, 2}, {1, 3}, {1, 4}, {1, 5}, {2, 3}, {2, 4}, {2, 5}, {3, 4}, {3, 5}, {4, 5}]
h_values_Jappx3 =[set(), {1}, {2}, {3}, {4},{1, 2}, {1, 3}, {1, 4}, {2, 3}, {2, 4}, {3, 4},{1, 2, 3}, {1, 2, 4}, {1, 3, 4}, {2, 3, 4}]
# h_values_Jappx3 =[set(), {1}, {2}, {3}, {4}, {5},{1, 2}, {1, 3}, {1, 4}, {1, 5}, {2, 3}, {2, 4}, {2, 5}, {3, 4}, {3, 5}, {4, 5},{1, 2, 3}, {1, 2, 4}, {1, 2, 5}, {1, 3, 4}, {1, 3, 5}, {1, 4, 5}, {2, 3, 4}, {2, 3, 5}, {2, 4, 5}, {3, 4, 5}]
# h_values_Jappx_All = [h_values_Jappx0,h_values_Jappx1,h_values_Jappx2]
h_values_Jappx_All = [h_values_Jappx0,h_values_Jappx1,h_values_Jappx2,h_values_Jappx3]

In [8]:
gamma=0
t_max = 1200
dt=0.1
t_values = np.arange(0, t_max+dt, dt)
n_steps = len(t_values)
# t_emerge=[0,0,0,0,0]
I0_val=list(0.0005*np.ones(len(nu_values)))
S0 = 1-sum(I0_val)

r_beta=np.ones(len(nu_values))
R0=2.5
R_nu = np.repeat(r_beta * R0,n_steps).reshape((len(r_beta),n_steps))

# mean_gentime=5
mean_gentime=7
gen_dist_params = [mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2),
                   mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2),
                   mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2),
                   mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2),
                   mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2)]

gamma_dist_params = [mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2),
                   mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2),
                   mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2),
                   mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2),
                   mean_sd_to_alpha_scale_gamma_dist(mean_gentime,2)]

g_table = np.array([[g_nu_h(tau=t_values,alpha=gen_dist_params[i][0], scale=gen_dist_params[i][1],nu=nu,h=h) for k, h in enumerate(h_values_all) ]
                    for i, nu in enumerate(nu_values)])
gamma_table = np.array([[gamma_u_h(tau=t_values,alpha=gamma_dist_params[i][0], scale=gamma_dist_params[i][1],nu=nu,h=h) for k, h in enumerate(h_values_all) ]
                        for i, nu in enumerate(nu_values)])
survivial_table = np.array([[l_nu_h2(tau=t_values,alpha=gamma_dist_params[i][0], scale=gamma_dist_params[i][1],nu=nu,h=h) for k, h in enumerate(h_values_all) ]
                        for i, nu in enumerate(nu_values)])


In [13]:
sigma=0.4
sigma_pair = sigma*(np.ones((len(nu_values),len(nu_values)))-np.eye(len(nu_values)))
sigma_nu_h = init_sigma_nu_h(sigma_pair,nu_values,h_values_all,method='min')
tilde_sigma_nu_h = np.ones((len(nu_values), len(h_values_all)))

## Single-run solver comparison

With the model parameters initialized, these cells run the exact solver and selected approximations for one fixed emergence interval. They prepare the initial infection terms and collect the resulting incidence, infection, and susceptibility trajectories for comparison before the parameter sweep.


In [9]:
deltate=200
t_emerge=np.arange(0,len(nu_values))*deltate
t_emerge_indx=[int(t_emerge[i]/dt) for i in range(len(t_emerge))]
IntgImpG_Emerg = np.array([np.array([IntgImptG1_t(tau=t_values-t_emerge[i],alpha=gen_dist_params[i][0], scale=gen_dist_params[i][1],nu=nu,h=h,I0=I0_val[i]) for k, h in enumerate(h_values_all) ]).sum(0)
                                       for i, nu in enumerate(nu_values)])

IntgImpGamma_Emerg = np.array([[IntgImptGamma1_t(t_values-t_emerge[i],alpha=gamma_dist_params[i][0], scale=gamma_dist_params[i][1],nu=nu,h=h,I0=I0_val[i]) for k, h in enumerate(h_values_all) ] 
                               for i, nu in enumerate(nu_values)])

s_values = np.zeros((len(h_values_all), n_steps))
s_values[h_index_map[tuple({})], :] = S0
# J_values = IntgImpG_Emerg.copy()
J_values = np.zeros_like(IntgImpG_Emerg)
t_emerge

array([  0, 200, 400, 600])

In [16]:
%%time
infectionRenewalEq_x86_v3.time_evolution_coupled(
        t_values,J_values, s_values,
        R_nu, sigma_nu_h, tilde_sigma_nu_h,
        g_table, gamma_table, IntgImpG_Emerg, IntgImpGamma_Emerg,
        nu_values, h_values_all,
        h_index_map)

J_values_ex=J_values.copy()
s_values_ex=s_values.copy()
s_eff_ex = sigma_nu_h@s_values_ex
I_values_ex=(s_values*sigma_nu_h[:, :, np.newaxis]*J_values[:, np.newaxis, :])
Inf_values_ex =np.array([[fftconvolve(I_values_ex[i, j], survivial_table[i, j], mode='full')[0:len(t_values)]*dt 
                           for j in range(len(h_values_all)) ] for i in range(len(nu_values))])
I_nu_ex=I_values_ex.sum(1) 
Inf_nu_ex=Inf_values_ex.sum(1) 

CPU times: user 8.81 s, sys: 163 ms, total: 8.97 s
Wall time: 8.82 s


In [15]:
%%time
h_values_Jappx = h_values_Jappx_All[0]
h_values_appx = [x for x in h_values_all if x not in h_values_Jappx]

s_values = np.zeros((len(h_values_all), n_steps))
J_values = IntgImpG_Emerg.copy()
s_values[h_index_map[tuple({})], :] = S0

infectionRenewalEq_x86_v3.time_evolution_coupled(
            t_values, J_values, s_values,
            R_nu, sigma_nu_h, tilde_sigma_nu_h,
            g_table, gamma_table, IntgImpG_Emerg, IntgImpGamma_Emerg,
            nu_values, h_values_Jappx,
            h_index_map )

infectionRenewalEq_x86_v3.time_evolution_uncoupled(
            t_values, J_values, s_values,
            R_nu, sigma_nu_h, tilde_sigma_nu_h,
            g_table, gamma_table, IntgImpG_Emerg, IntgImpGamma_Emerg,
            nu_values, h_values_all, h_values_appx,
            h_index_map )

J_values_appx0=J_values.copy()
s_values_appx0=s_values.copy()
s_eff_appx0 = sigma_nu_h@s_values_appx0
I_values_appx0=(s_values_appx0*sigma_nu_h[:, :, np.newaxis]*J_values_appx0[:, np.newaxis, :])
Inf_values_appx0 =np.array([[fftconvolve(I_values_appx0[i, j], survivial_table[i, j], mode='full')[0:len(t_values)]*dt 
                           for j in range(len(h_values_all)) ] for i in range(len(nu_values))])
I_nu_appx0=I_values_appx0.sum(1) 
Inf_nu_appx0=Inf_values_appx0.sum(1)

CPU times: user 3.97 s, sys: 108 ms, total: 4.08 s
Wall time: 3.78 s


In [17]:
%%time
s_values = np.zeros((len(h_values_all), n_steps))
s_values[h_index_map[tuple({})], :] = S0
# J_values = IntgImpG_Emerg.copy()
J_values = np.zeros_like(IntgImpG_Emerg)
n_overlap=1
Lambda_approx_1cir, s_approx_1cir = infectionRenewalEq_seqcirc_approx.time_evolution_seqcirc_approx(
    t_values,
    J_values,           # seed import term (= IntgImpG_Emerg)
    s_values,    # pass a copy so we can compare with exact later
    R_nu,
    sigma_nu_h,
    tilde_sigma_nu_h,
    g_table,
    gamma_table,
    IntgImpG_Emerg,
    IntgImpGamma_Emerg,
    nu_values,
    h_values_all,
    h_index_map,
    list(t_emerge),     # T^em_c for each variant
    n_overlap
)

s_eff_appSeq_1cir = sigma_nu_h@s_approx_1cir
I_values_appSeq_1cir=(s_approx_1cir*sigma_nu_h[:, :, np.newaxis]*Lambda_approx_1cir[:, np.newaxis, :])
Inf_values_appSeq_1cir =np.array([[fftconvolve(I_values_appSeq_1cir[i, j], survivial_table[i, j], mode='full')[0:len(t_values)]*dt 
                           for j in range(len(h_values_all)) ] for i in range(len(nu_values))])
I_nu_appSeq_1cir=I_values_appSeq_1cir.sum(1) 
Inf_nu_appSeq_1cir=Inf_values_appSeq_1cir.sum(1) 

CPU times: user 997 ms, sys: 111 ms, total: 1.11 s
Wall time: 780 ms


## Parameter sweep and derived results

The run_simulation function evaluates the exact model and its approximations for one emergence schedule. The sweep varies emergence spacing and sigma, then stores the results. The following cells load a saved result array and derive incidence, susceptibility, effective reproduction numbers, and relative errors for plotting.


In [ ]:
def run_simulation(t_emerge):
    results_local = {}

    s_values = np.zeros((len(h_values_all), n_steps))
    J_values = IntgImpG_Emerg.copy()
    s_values[h_index_map[tuple({})], :] = S0

    infectionRenewalEq_x86_v3.time_evolution_coupled(
        t_values, J_values, s_values,
        R_nu, sigma_nu_h, tilde_sigma_nu_h,
        g_table, gamma_table, IntgImpG_Emerg, IntgImpGamma_Emerg,
        nu_values, h_values_all,
        h_index_map )
    results_local['par'] = {'sigma': sigma }
    results_local['ex'] = {'s_values': np.array(s_values, copy=True),
                           'J_values': np.array(J_values, copy=True) }

    s_values = np.zeros((len(h_values_all), n_steps))
    s_values[h_index_map[tuple({})], :] = S0
    J_values = np.zeros_like(IntgImpG_Emerg)
    n_overlap=1
    Lambda_approx_seq_1cir, s_approx_seq_1cir = infectionRenewalEq_seqcirc_approx.time_evolution_seqcirc_approx(
        t_values, J_values, s_values, 
        R_nu, sigma_nu_h, tilde_sigma_nu_h,
        g_table, gamma_table, IntgImpG_Emerg, IntgImpGamma_Emerg,
        nu_values, h_values_all, h_index_map,
        list(t_emerge), n_overlap)

    results_local['appx_seq_1cir'] = {
            's_values': np.array(s_approx_seq_1cir, copy=True),
            'J_values': np.array(Lambda_approx_seq_1cir, copy=True) }

    s_values = np.zeros((len(h_values_all), n_steps))
    s_values[h_index_map[tuple({})], :] = S0
    J_values = np.zeros_like(IntgImpG_Emerg)
    n_overlap=2
    Lambda_approx_seq_2cir, s_approx_seq_2cir = infectionRenewalEq_seqcirc_approx.time_evolution_seqcirc_approx(
        t_values, J_values, s_values, 
        R_nu, sigma_nu_h, tilde_sigma_nu_h,
        g_table, gamma_table, IntgImpG_Emerg, IntgImpGamma_Emerg,
        nu_values, h_values_all, h_index_map,
        list(t_emerge), n_overlap)

    results_local['appx_seq_2cir'] = {
            's_values': np.array(s_approx_seq_2cir, copy=True),
            'J_values': np.array(Lambda_approx_seq_2cir, copy=True) }
    
    for appx in range(len(h_values_Jappx_All)):
        h_values_Jappx = h_values_Jappx_All[appx]
        h_values_appx = [x for x in h_values_all if x not in h_values_Jappx]

        s_values = np.zeros((len(h_values_all), n_steps))
        J_values = IntgImpG_Emerg.copy()
        s_values[h_index_map[tuple({})], :] = S0

        infectionRenewalEq_x86_v3.time_evolution_coupled(
            t_values, J_values, s_values,
            R_nu, sigma_nu_h, tilde_sigma_nu_h,
            g_table, gamma_table, IntgImpG_Emerg, IntgImpGamma_Emerg,
            nu_values, h_values_Jappx,
            h_index_map )

        infectionRenewalEq_x86_v3.time_evolution_uncoupled(
            t_values, J_values, s_values,
            R_nu, sigma_nu_h, tilde_sigma_nu_h,
            g_table, gamma_table, IntgImpG_Emerg, IntgImpGamma_Emerg,
            nu_values, h_values_all, h_values_appx,
            h_index_map )

        appx_key = f'appx_{appx}'
        results_local[appx_key] = {
            's_values': np.array(s_values, copy=True),
            'J_values': np.array(J_values, copy=True) }

    return results_local

In [ ]:
%%time
# delta_t_values = [0, 5, 10, 20, 30, 40, 50, 100, 150, 200, 250]
# delta_t_values = [0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240]
# delta_t_values = [0, 40, 60, 80, 120, 140, 160, 180, 220, 240]
delta_t_values = [15, 25]

sigma_values = np.arange(0,1.1,0.1)

results_t = []
for deltate in delta_t_values:
    t_emerge=np.arange(0,5)*deltate
    t_emerge_indx=[int(t_emerge[i]/dt) for i in range(len(t_emerge))]
    
    IntgImpG_Emerg = np.array([np.array([IntgImptG1_t(tau=t_values-t_emerge[i],alpha=gen_dist_params[i][0], scale=gen_dist_params[i][1],nu=nu,h=h,I0=I0_val[i]) for k, h in enumerate(h_values_all) ]).sum(0)
                                           for i, nu in enumerate(nu_values)])
    IntgImpGamma_Emerg = np.array([[IntgImptGamma1_t(t_values-t_emerge[i],alpha=gamma_dist_params[i][0], scale=gamma_dist_params[i][1],nu=nu,h=h,I0=I0_val[i]) for k, h in enumerate(h_values_all) ] 
                                   for i, nu in enumerate(nu_values)])
    
    results = []
    mes = "Computing for delta temrg = "+str(deltate)+" and sigma"
    for sigma in tqdm(sigma_values, desc=mes):
        
        sigma_pair = sigma*(np.ones((len(nu_values),len(nu_values)))-np.eye(len(nu_values)))
        sigma_nu_h = init_sigma_nu_h(sigma_pair,nu_values,h_values_all,method='min')
        s_values = np.zeros((len(h_values_all), n_steps))
        s_values[h_index_map[tuple({})], :] = S0
        J_values = IntgImpG_Emerg.copy()
        res = run_simulation(t_emerge=t_emerge)
        results.append(res)

    results_t.append(results) 

In [ ]:
# np.save("../data/ode_result_delta2_15_25.npy", results_t, allow_pickle=True)
# np.save("../data/my_data.npy", results_t, allow_pickle=True)
# np.save("../data/my_data_newdeltat.npy", results_t, allow_pickle=True)
# np.save("../data/ode_results_DeltaT_0_250.npy", combined_results, allow_pickle=True)

# delta_t_values = [0,5,10,20,30,40,50,100,150,200,250]
# delta_t_values_new = [0, 40, 60, 80, 120, 140, 160, 180, 220, 240]
delta_t_values_new = [15,25]
delta_t_values = [0,   5,  10,  20,  30,  40,  50,  60,  80, 100, 120, 140, 150, 160, 180, 200, 220, 240, 250]


# results_t_new =np.load("../Data/ode_resultDeltaT_15_25.npy", allow_pickle=True)
# results_t =np.load("../Data/ode_results_DeltaT_0_250.npy", allow_pickle=True)

# delta_t_values_merged = [ 0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 140, 150, 160, 180, 200, 220, 240, 250]
# data_by_dt = dict(zip(delta_t_values, results_t))
# data_by_dt.update(dict(zip(delta_t_values_new, results_t_new)))
# merged_list = [ data_by_dt[dt] for dt in delta_t_values_merged ]

# np.save("../data/ode_results_combinedAllDeltaT.npy", merged_list, allow_pickle=True)


In [ ]:
sigma_values = np.arange(0,1.1,0.1)
delta_t_values = [ 0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 140, 150, 160, 180, 200, 220, 240, 250]

results_t =np.load("../Data/ode_results_combinedAllDeltaT.npy", allow_pickle=True)
len(results_t)

In [ ]:
methods = [
    "ex",
    "appx_seq_1cir",
    "appx_seq_2cir",
    "appx_0",
    "appx_1",
    "appx_2"
]
h_indices_trucnated = {'ex' : [i for i, h in enumerate(h_values_all)],
    'appx_seq_1cir' : [i for i, h in enumerate(h_values_all)],
    'appx_seq_2cir' : [i for i, h in enumerate(h_values_all)],
    'appx_0' : [i for i, h in enumerate(h_values_all)  if h in h_values_Jappx0 ], 
    'appx_1' : [i for i, h in enumerate(h_values_all)  if h in h_values_Jappx1 ], 
    'appx_2' : [i for i, h in enumerate(h_values_all)  if h in h_values_Jappx2 ]}


def calculate_results(results_t, delta_t_values, sigma_values,
                      nu_values, h_values_all,h_indices_trucnated,methods):

    
    n_dt = len(delta_t_values)
    n_sigma = len(sigma_values)
    
    # Get dimensions
    s_shape = results_t[0][0]["ex"]["s_values"].shape
    j_shape = results_t[0][0]["ex"]["J_values"].shape

    n_h, n_t = s_shape
    n_nu, _ = j_shape

    # Preallocate
    Inc = {}
    sh = {}
    J = {}
    seff = {}
    seff_truncated = {}

    for method in methods:

        Inc[method] = np.empty((n_dt, n_sigma, n_nu, n_h, n_t), dtype=np.float64 )
        sh[method] = np.empty((n_dt, n_sigma, n_h, n_t), dtype=np.float64 )
        J[method] = np.empty((n_dt, n_sigma, n_nu, n_t), dtype=np.float64 )
        seff[method] = np.empty((n_dt, n_sigma, n_nu, n_t), dtype=np.float64 )
        seff_truncated[method] = np.empty((n_dt, n_sigma, n_nu, n_t), dtype=np.float64 )
        

    # Precompute the off-diagonal mask
    sigma_mask = ( np.ones((n_nu, n_nu))- np.eye(n_nu) )

    for te_ind in range(n_dt):

        results = results_t[te_ind]

        for s_ind, sigma in enumerate(sigma_values):

            sigma_pair = sigma * sigma_mask
            sigma_nu_h = init_sigma_nu_h( sigma_pair,nu_values,h_values_all, method="min" )

            for method in methods:

                result = results[s_ind][method]
                s = result["s_values"]
                j = result["J_values"]
                # Store J
                J[method][te_ind, s_ind] = j
                # Store susceptibility
                sh[method][te_ind, s_ind] = s
                # Effective susceptibility
                seff[method][te_ind, s_ind] = sigma_nu_h @ s
                mask_sh_appx =  np.zeros_like(s)
                mask_sh_appx[h_indices_trucnated[method],] = 1
                seff_truncated[method][te_ind, s_ind] = sigma_nu_h @ (s*mask_sh_appx)
                # Incidence
                Inc[method][te_ind, s_ind] = s[:, :] * sigma_nu_h[:, :, np.newaxis] * j[:, np.newaxis, :] 

            del sigma_pair, sigma_nu_h

    return Inc, J, sh, seff, seff_truncated

In [ ]:
Inc, J, sh, seff, seff_truncated = calculate_results(results_t,delta_t_values,sigma_values, nu_values, h_values_all,h_indices_trucnated, methods)

In [ ]:
def compute_Rt_arrays(Inc_arr, sh_arr, seff_arr, h_indices_keep=[]):
    Rt_nu = []
    Rt = []
    for te_ind in range(len(delta_t_values)):
        Rt_nu_tmp = []
        Rt_tmp = []
        for s_ind, sigma in enumerate(sigma_values):
            sigma_pair = sigma * (np.ones((len(nu_values), len(nu_values))) - np.eye(len(nu_values)))
            sigma_nu_h = init_sigma_nu_h(sigma_pair, nu_values, h_values_all, method='min')

            s_val_tmp = sh_arr[te_ind, s_ind]
            s_eff_tmp = seff_arr[te_ind, s_ind]
            Inc_nu_h = Inc_arr[te_ind, s_ind]
            if len(h_indices_keep)>0 : 
                h_mask = np.ones(len(h_values_all), dtype=bool)
                h_mask[h_indices_keep] = False
                s_val_tmp[h_mask,:] = 0
                Inc_nu_h[:,h_mask,:] = 0

            Inc_nu_emp = Inc_nu_h.sum(1)
            h_tmp = Inc_nu_emp / Inc_nu_emp.sum(0)[np.newaxis, :]
            h_tmp[0, 0] = 1
            h_tmp[1:, 0] = 0

            R_nu_tmp = ( np.array(R_nu) * s_eff_tmp) * np.array([[myconvolution_fft(
                        g=g_table[i, j],
                        f=tilde_sigma_nu_h[i, j] * sigma_nu_h[i, j] * s_val_tmp[j] / s_eff_tmp[i], hind=j) * dt 
                        for j in range(len(h_values_all))] for i in range(len(nu_values)) ]).sum(1)

            R_tmp = ((np.array(R_nu) * s_eff_tmp)* (np.array([myconvolution_fft(g=g_table[i, 0],f=h_tmp[i],hind=1) * dt
                        for i in range(len(nu_values))]) / (g_table[:, 0].cumsum(1) * dt) )).sum(0)

            Rt_nu_tmp.append(R_nu_tmp)
            Rt_tmp.append(R_tmp)

        Rt_nu.append(Rt_nu_tmp)
        Rt.append(Rt_tmp)

    return np.array(Rt_nu), np.array(Rt)

In [ ]:
method_names = [
    "ex",
    "appx_0",
    "appx_1",
    "appx_2",
    "appx_seq_1cir",
    "appx_seq_2cir"
]

h_indices_trucnated = {
    'ex' : [i for i, h in enumerate(h_values_all)],
    'appx_seq_1cir' : [i for i, h in enumerate(h_values_all)],
    'appx_seq_2cir' : [i for i, h in enumerate(h_values_all)],
    'appx_0' : [i for i, h in enumerate(h_values_all)  if h in h_values_Jappx0 ], 
    'appx_1' : [i for i, h in enumerate(h_values_all)  if h in h_values_Jappx1 ], 
    'appx_2' : [i for i, h in enumerate(h_values_all)  if h in h_values_Jappx2 ]}

Rt_nu = {}
Rt = {}

for method in method_names:
    # mask_sh_appx =  np.zeros_like(sh[method])
    # mask_sh_appx[h_indices_trucnated[method],] = 1
    Rt_nu[method], Rt[method] = compute_Rt_arrays(
        Inc[method],
        sh[method],
        seff_truncated[method], 
        h_indices_keep=h_indices_trucnated[method]
    )

In [ ]:
from scipy.signal import find_peaks

def last_peak_along_time(arr):
    *shape, T = arr.shape
    arr_flat = arr.reshape(-1, T)

    last_peaks = []

    for row in arr_flat:
        peaks, _ = find_peaks(row)
        last_peaks.append(peaks[-1] if len(peaks) > 0 else np.nan)

    return np.array(last_peaks).reshape(shape)

In [ ]:
method_names = [
    "appx_0",
    "appx_1",
    "appx_2",
    "appx_seq_1cir",
    "appx_seq_2cir"
]

Inc_ex_var  = Inc["ex"].sum(axis=3)
Inc_ex_total  = Inc_ex_var.sum(axis=2)

mask = np.arange(Inc["ex"].shape[-1]) <= (last_peak_along_time(Inc["ex"].sum(axis=3))[...,None])

Error_r_Var = []
Error_abs_Var = []
Error_r_Total = []
Error_abs_Total = []
Error_r_Var_uptopeak = []

for method in method_names:
    # Sum over h
    h_ind_includ = h_indices_trucnated[method]
    Inc_app_var = Inc[method][:,:,:,h_ind_includ,:].sum(axis=3)
    # Sum over nu
    Inc_app_total = Inc_app_var.sum(axis=2)
    # variant error
    error_var =   (np.abs(Inc_app_var - Inc_ex_var).sum(axis=3) * dt )
    error_var_r = (error_var / (Inc_ex_var.sum(axis=3) * dt) )
    Error_abs_Var.append(error_var)
    Error_r_Var.append(error_var_r)
    # total error
    error_tot = (np.abs(Inc_app_total - Inc_ex_total).sum(axis=2) * dt)
    error_tot_r = (error_tot/ (Inc_ex_total.sum(axis=2) * dt) )
    Error_abs_Total.append(error_tot)
    Error_r_Total.append(error_tot_r)
    # tovariant error up to peak
    error_var_uptopeak = np.sum(np.abs(Inc_app_var - Inc_ex_var) * mask, axis=3)
    Error_r_Var_uptopeak.append(error_var_uptopeak / np.sum(Inc_ex_var * mask, axis=3) )

Error_abs_Var = np.array(Error_abs_Var)
Error_r_Var = np.array(Error_r_Var)
Error_r_Total = np.array(Error_r_Total)
Error_abs_Total = np.array(Error_abs_Total)
Error_r_Var_uptopeak = np.array(Error_r_Var_uptopeak)

Error_r = np.concatenate([Error_r_Var, Error_r_Total[..., np.newaxis] ], axis=-1 )
Error_abs = np.concatenate([Error_abs_Total[..., np.newaxis], Error_abs_Var], axis=-1)

Error_r_J_Var = []
for method in method_names:
    error = (np.abs(J[method] - J["ex"]).sum(axis=3) * dt/( J["ex"].sum(axis=3) * dt))
    Error_r_J_Var.append(error)

Error_r_J_Var = np.array(Error_r_J_Var)


In [ ]:
method_names = [
    "appx_0",
    "appx_1",
    "appx_2",
    "appx_seq_1cir",
    "appx_seq_2cir"
]
mask_cir1 = Inc["appx_seq_1cir"].sum(axis=3) != 0
mask_cir2 = Inc["appx_seq_2cir"].sum(axis=3) != 0

Error_r_Var_masked_cir = []
for method in method_names:
    Inc_app_var = Inc[method].sum(axis=3)
    Inc_ex_var  = Inc["ex"].sum(axis=3)
    if method == "appx_seq_1cir":
        mask = mask_cir1
    elif method == "appx_seq_2cir":
        mask = mask_cir2
    else:
        mask = np.ones_like(Inc_app_var, dtype=bool)

    error = ( np.sum( np.abs(Inc_app_var - Inc_ex_var) * mask, axis=3 ) / np.sum(Inc_ex_var * mask, axis=3))
    Error_r_Var_masked_cir.append(error)

Error_r_Var_masked_cir = np.array(Error_r_Var_masked_cir)
Error_r_masked_cir = np.concatenate([Error_r_Var_masked_cir, Error_r_Total[..., np.newaxis]],axis=-1)

In [ ]:
Error_r_Var_uptopeak_mased_cir = np.concatenate([
    Error_r_Var_uptopeak[[0,1,2], :, :, :],
    Error_r_masked_cir[[3,4], :, :, 0:4]
], axis=0)

# np.save("../data/Error_r_Var_uptopeak.npy",Error_r_Var_uptopeak_mased_cir )
# np.save("../data/Error_r_Var_uptopeak_truncatedH.npy",Error_r_Var_uptopeak_mased_cir )


## Plot and compare results

The final section plots incidence and susceptibility trajectories, reproduction-number comparisons, and error heatmaps across the selected parameters. Figure exports use ../figures/; the optional summary table is written to ../data/.


In [ ]:
import matplotlib.patheffects as pe

sigma_sel = 0.5
tem_sel=[0,10,20,50,100,200]
# tem_sel=[0,20,25,40,60,80,100,120,140,160]

sigma_ind = np.argmin(np.abs(sigma_values - sigma_sel))
tem_list = [np.where(np.array(delta_t_values)==t)[0][0] for t in tem_sel]
nu_list = range(4)
# =========================
# METHODS TO PLOT
# =========================
method_label = {
    "ex": "Exact",
    "appx_0": "Zero order truncation",
    "appx_1": "1st order truncation",
    "appx_2": "2nd order truncation",
    "appx_seq_1cir": "One variants at the time",
    "appx_seq_2cir": "Two variants at the time"
}
h_indices_trucnated = {
    "ex" : [i for i, h in enumerate(h_values_all)],
    "appx_seq_1cir" : [i for i, h in enumerate(h_values_all)],
    'appx_seq_2cir' : [i for i, h in enumerate(h_values_all)],
    "appx_0" : [i for i, h in enumerate(h_values_all)  if h in h_values_Jappx0 ], 
    "appx_1" : [i for i, h in enumerate(h_values_all)  if h in h_values_Jappx1 ], 
    'appx_2' : [i for i, h in enumerate(h_values_all)  if h in h_values_Jappx2 ]}

methods = {
    "ex": Inc["ex"],
    "appx_0": Inc["appx_0"],
    "appx_1": Inc["appx_1"],
    "appx_2": Inc["appx_2"],
    "appx_seq_1cir": Inc["appx_seq_1cir"],
    "appx_seq_2cir": Inc["appx_seq_2cir"]
}

strain_labels = ['Variant 1 (Ancestral Virus)', 'Variant 2', 'Variant 3', 'Variant 4', 'Total']
# strain_labels = ['Ancestral Virus', 'Variant 1', 'Variant 2', 'Variant 3']
colors = {
    "ex": "#362C1E",
    "appx_0": "#E9C46A",   
    "appx_1": "#F09729",  
    "appx_2": "#C84E13",
    # circulating family (purple scale, clearly distinct shades)
    "appx_seq_1cir": "#B794F4" , 
    "appx_seq_2cir": "#5B21B6" 
}

linestyles = [ "-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1)),]
lw = {
    "ex": 4.5,
    "appx_0": 2.5,
    "appx_1": 3,
    "appx_2": 3.5,
    "appx_seq_1cir": 1.5,
    "appx_seq_2cir": 2}

markers = {
    "ex": "o",
    "appx_0": "s",
    "appx_1": "^",
    "appx_2": "D",
    "appx_seq_1cir": "P",
    "appx_seq_2cir": "X"}

nrows = len(tem_list)
ncols = 4   # 4 nus 
# ncols = 5   # 4 nus + total
fig, axes = plt.subplots( nrows=nrows, ncols=ncols, figsize=(4*ncols, 3*nrows), sharex=False,)

# if only one row
if nrows == 1:
    axes = axes[np.newaxis, :]

methods_list = [list(methods.items())[i] for i in [0,3,2,1,5,4]]

for row, tem_ind in enumerate(tem_list):
    t_emerge=np.arange(0,len(nu_values))*delta_t_values[tem_ind]
    t_emerge_indx=[int(t_emerge[i]/dt) for i in range(len(t_emerge))]   
    # -----------------------------------
    # columns 1-4 : individual nu
    for col, nu in enumerate(nu_list):
        ax = axes[row, col]
        eps = 1e-4
        y_ex=Inc["ex"][tem_ind, sigma_ind, nu].sum(0)
        cut = np.where(y_ex > eps)[0]
        if len(cut) > 0:
            end = cut[-1] + 1
        else:
            end = len(t_values)

        y_ex=Inc["ex"][tem_ind, sigma_ind, nu].sum(0)
        ymax = np.nanmax(y_ex)
        ymin = np.nanmin(y_ex)
        ymin = max(ymin, 1e-6)
        # ax.set_ylim([ymin*0.8, ymax*1.3])
       
        strt = t_emerge_indx[nu]
        for k, (method, DT) in enumerate(methods_list):
            # y=DT[tem_ind, sigma_ind, nu].sum(0)
            y=DT[tem_ind, sigma_ind, nu][h_indices_trucnated[method]].sum(0)
           
            if (row==0 and k>3): continue
            y[y == 0] = np.nan
            # ax.plot(t_values[:end],abs(y_ex[:end]-y[:end]), color=colors[method], linestyle=linestyles[k], linewidth=2,label=method_label[method])
            line, = ax.plot(t_values[:end],y[:end], color=colors[method], linestyle=linestyles[0], linewidth=lw[method],
                            marker=markers[method],alpha=0.9, markevery=int((end-strt)/15),markersize=6,label=method_label[method])
            # line.set_path_effects([pe.Stroke(linewidth=1, foreground='white',alpha=0.9),pe.Normal()])

        ax.set_ylim([-0.05 * ymax, ymax*1.25])
        ax.set_ymargin(m=0.05)
        # ax.set_yscale('log') 
        # ax.set_ylim([10**-4, 10**-1])
        if row == 0:
            # ax.set_title(fr"$\sigma = {round(sigma_sel,2)},\ \Delta t = {delta_t_values[tem_ind]}$, {strain_labels[nu]}")
            ax.set_title(fr"{strain_labels[nu]}",fontsize=14)

        if col == 0:
            ax.set_ylabel("$\Delta t = " + str(delta_t_values[tem_ind]) + "$",fontsize=14,labelpad=30, rotation=0)
        ax.grid(alpha=0.3)

    # -----------------------------------
    # 5th column : total over nu
    if ncols > 4:
        ax = axes[row, 4]
        for k, (method, DT) in enumerate(methods.items()):
            if (row==0 and k>3): continue
            # y = DT[tem_ind, sigma_ind, :].sum(0).sum(0)
            y = DT[tem_ind, sigma_ind, :][h_indices_trucnated[method]].sum(0).sum(0)
            # y_ex = Inc_ex[tem_ind, sigma_ind, :].sum(0).sum(0)
            
            # ax.plot( t_values[:end],abs(y_ex[:end]-y[:end]), color=colors[method], linestyle=linestyles[k], linewidth=2, label=method_label[method],)
            # ax.plot( t_values[:end],IntgImpGamma_Emerg.sum(0).sum(0)[:end], color='red', linestyle=linestyles[k], linewidth=2, label=method_label[method],)
            ax.plot( t_values[:(end-100)],y[:(end-100)], color=colors[method], linestyle=linestyles[0], linewidth=lw[method], 
                    marker=markers[method],markersize=6,alpha=0.95, markevery=int(end/20)+1, label=method_label[method])

        # ax.set_title(fr"$\sigma = {round(sigma_sel,2)},\ \Delta t = {delta_t_values[tem_ind]}$, Total")
        if row == 0:
            ax.set_title(fr"Total",fontsize=14)

        # ax.set_yscale('log') 
        # ax.set_ylim([10**-6, 10**-1])
        
        ax.grid(alpha=0.3)

# =========================
# COMMON LEGEND
# =========================
fig.supxlabel(r"time (days)",fontsize=18)
fig.supylabel("Incidence",fontsize=18)
handles, labels = axes[1,0].get_legend_handles_labels()
fig.legend( handles, labels, loc="upper center", ncol=3, fontsize=16, frameon=False,)
plt.tight_layout(rect=[0, 0, 1, 0.95])
# fig.savefig("../figures/" + ('Inc_TotalandAllStrain_Ex_AppxAndCircAppx_Deltat'+'_Sigma='+str(round(sigma_sel,2))+'_newPar_Rr0=2.5.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('Inc_AllStrain_Ex_AppxAndCircAppx_Deltat'+'_Sigma='+str(round(sigma_sel,2))+'_newPar_Rr0=2.5.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('Inc_AllStrain_Ex_AppxAndCircAppx_Deltat'+'_Sigma='+str(round(sigma_sel,2))+'_newPar_Rr0=2.5_truncatedH.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('Inc_TotalandAllStrain_Ex_AppxAndCircAppx_Deltat'+'_Sigma='+str(round(sigma_sel,2))+'_log.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()  

del DT 

In [ ]:
import matplotlib.patheffects as pe

sigma_sel = 0.5
tem_sel=[0,10,20,50,100,200]
# tem_sel=[0,20,40,60,80,100,120,140,160]

sigma_ind = np.argmin(np.abs(sigma_values - sigma_sel))
tem_list = [np.where(np.array(delta_t_values)==t)[0][0] for t in tem_sel]
nu_list = range(4)
# =========================
# METHODS TO PLOT
# =========================
methods = {
    "Exact": seff["ex"],
    "Zero order truncation": seff["appx_0"],
    "1st order truncation": seff["appx_1"],
    "2nd order truncation": seff["appx_2"],
    "One variants at the time": seff["appx_seq_1cir"],
    "Two variants at the time": seff["appx_seq_2cir"]
}

strain_labels = ['Variant 1 (Ancestral Virus)', 'Variant 2', 'Variant 3', 'Variant 4', 'Total']
# strain_labels = ['Ancestral Virus', 'Variant 1', 'Variant 2', 'Variant 3']
# colors=['#362C1E','#8D7659','#F06816','#F09729','#F0B73F','#C94C10']
# colors=[ '#362C1E','#F0B73F','#F09729','#F06816','#8D7659',"tab:purple"]
colors = {
    "Exact": "#362C1E",
    # Taylor / perturbation hierarchy (warm scale)
    # "Zero order": "#F0B73F",
    # "1st order": "#F09729",
    # "2nd order": "#F06816",
    "Zero order truncation": "#E9C46A",   # sand gold
    "1st order truncation": "#F09729",  
    "2nd order truncation": "#C84E13",

    # circulating family (purple scale, clearly distinct shades)
    "One variants at the time": "#B794F4" ,   # "A78BFA"soft violet
    "Two variants at the time": "#5B21B6" #"#6D28D9"    # strong violet (not too dark)
}
# colors = ["black", "tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
linestyles = [ "-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1)),]
lw = [3.5,2.0, 2.5, 3.0, 1,1.5]
lw=[3.5,3,2.5,2,1.5,1]
lw=[5,3.5,3,2.5,2,1.5]
mrks=[9,8,7,6,5,4]


markers = {
    "Exact": "o",
    "Zero order truncation": "s",
    "1st order truncation": "^",
    "2nd order truncation": "D",
    "One variants at the time": "P",
    "Two variants at the time": "X"}
nrows = len(tem_list)
ncols = 4   # 4 nus 
fig, axes = plt.subplots( nrows=nrows, ncols=ncols, figsize=(4*ncols, 3*nrows), sharex=False,)

# if only one row
if nrows == 1:
    axes = axes[np.newaxis, :]

methods_list = [list(methods.items())[i] for i in [0,3,2,1,5,4]]

for row, tem_ind in enumerate(tem_list):
    t_emerge=np.arange(0,len(nu_values))*delta_t_values[tem_ind]
    t_emerge_indx=[int(t_emerge[i]/dt) for i in range(len(t_emerge))]   
    # -----------------------------------
    # columns 1-4 : individual nu
    for col, nu in enumerate(nu_list):
        ax = axes[row, col]
        eps = 1e-4
        y_ex=Inc["ex"][tem_ind, sigma_ind, nu].sum(0)
        cut = np.where(y_ex > eps)[0]
        if len(cut) > 0:
            end = cut[-1] + 1
        else:
            end = len(t_values)

        y_ex=Inc["ex"][tem_ind, sigma_ind, nu].sum(0)
        ymax = np.nanmax(y_ex)
        ymin = np.nanmin(y_ex)
        ymin = max(ymin, 1e-6)
        # ax.set_ylim([ymin*0.8, ymax*1.3])
       
        strt = t_emerge_indx[nu]
        for k, (label, DT) in enumerate(methods_list):
            y=DT[tem_ind, sigma_ind, nu]
           
            if (row==0 and k>3): continue
            line, = ax.plot(t_values[:end],y[:end], color=colors[label], linestyle=linestyles[0], linewidth=lw[k],
                            marker=markers[label],alpha=0.9, markevery=int((end-strt)/15),markersize=6,label=label)
            # line.set_path_effects([pe.Stroke(linewidth=1, foreground='white',alpha=0.9),pe.Normal()])

        # ax.set_ylim([-0.05 * ymax, ymax*1.25])
        # ax.set_ymargin(m=0.05)
        # ax.set_yscale('log') 
        # ax.set_ylim([10**-6, 10**-1])
        if row == 0:
            # ax.set_title(fr"$\sigma = {round(sigma_sel,2)},\ \Delta t = {delta_t_values[tem_ind]}$, {strain_labels[nu]}")
            ax.set_title(fr"{strain_labels[nu]}",fontsize=14)

        if col == 0:
            ax.set_ylabel("$\Delta t = " + str(delta_t_values[tem_ind]) + "$",fontsize=14,labelpad=30, rotation=0)
        ax.grid(alpha=0.3)


# =========================
# COMMON LEGEND
# =========================
fig.supxlabel(r"time (days)",fontsize=18)
fig.supylabel("Effective susceptibility",fontsize=18)
handles, labels = axes[1,0].get_legend_handles_labels()
fig.legend( handles, labels, loc="upper center", ncol=3, fontsize=16, frameon=False,)
plt.tight_layout(rect=[0, 0, 1, 0.95])

# fig.savefig("../figures/" + ('Eff_Sus_AllStrain_Ex_AppxAndCircAppx_Deltat'+'_Sigma='+str(round(sigma_sel,2))+'_log.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()  

del DT 

In [ ]:
# Plot Rt_nu_ex vs approximations (one panel per strain, rows = selected Delta t)
sigma_sel = 0.5
sigma_ind = np.argmin(np.abs(sigma_values - sigma_sel))
tem_sel = [0, 10, 20, 50, 100, 200]


tem_list = [np.where(np.array(delta_t_values) == t)[0][0] for t in tem_sel]
nu_list = range(len(nu_values))

strain_labels = ['Variant 1 (Ancestral Virus)', 'Variant 2', 'Variant 3', 'Variant 4', 'Total']

methods = {
    "Exact": Rt_nu['ex'],
    "Zero order truncation": Rt_nu['appx_0'],
    "1st order truncation": Rt_nu['appx_1'],
    "2nd order truncation": Rt_nu['appx_2'],
    "One variants at the time": Rt_nu['appx_seq_1cir'],
    "Two variants at the time": Rt_nu['appx_seq_2cir']
}
colors = plt.cm.tab10.colors
linestyles = ["-", "--", ":", "-.", (0, (5,1)), (0, (3,1,1,1))]

colors = {
    "Exact": "#362C1E",
    "Zero order truncation": "#E9C46A",   # sand gold
    "1st order truncation": "#F09729",  
    "2nd order truncation": "#C84E13",
    # circulating family (purple scale, clearly distinct shades)
    "One variants at the time": "#B794F4" ,   # "A78BFA"soft violet
    "Two variants at the time": "#5B21B6" #"#6D28D9"    # strong violet (not too dark)
}
linestyles = [ "-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1)),]
lw=[4.5,3.5,3,2.5,2,1.5]
mrks=[9,8,7,6,5,4]
markers = {
    "Exact": "o",
    "Zero order truncation": "s",
    "1st order truncation": "^",
    "2nd order truncation": "D",
    "One variants at the time": "P",
    "Two variants at the time": "X"}

nrows = len(tem_list)
ncols = len(nu_list)
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4*ncols, 3*nrows), sharex=False, sharey=False)


if nrows == 1:
    axes = axes[np.newaxis, :]

for row, tem_ind in enumerate(tem_list):
    t_emerge = np.arange(len(nu_values)) * delta_t_values[tem_ind]
    t_emerge_indx = [int(x / dt) for x in t_emerge]
    for col, nu in enumerate(nu_list):
        ax = axes[row, col]
        ts = t_emerge_indx[nu]
        # ts=0
        # plot each method
        for k, (label, arr) in enumerate(methods.items()):
            y = arr[tem_ind, sigma_ind, nu].copy()
            # y = np.array(y, copy=False)
            # y[y == 0] = np.nan
            ax.plot(t_values[ts:ts+3000], y[ts:ts+3000],
                    color=colors[label], linestyle=linestyles[0], linewidth=lw[k], 
                    marker=markers[label],markersize=6,alpha=0.95, markevery=int(end/20)+1, label=label)
        ax.axhline(1, color='gray', linestyle='--', linewidth=0.8)
        if row == 0:
            ax.set_title(f"{strain_labels[nu]}", fontsize=16)
        if col == 0:
            ax.set_ylabel(f"Δt = {delta_t_values[tem_ind]}", fontsize=14)
        ax.grid(alpha=0.3)
        # ax.set_xlim([t_values[ts], t_values[ts+3000]])
        # ax.set_ylim(bottom=0)
# common labels and legend
fig.supxlabel("time (days)", fontsize=18)
fig.supylabel("R(t)",fontsize=18)
handles, labels = axes[0,0].get_legend_handles_labels()
fig.legend( handles, labels, loc="upper center", ncol=3, fontsize=16, frameon=False,)

plt.tight_layout(rect=[0, 0, 1, 0.95])

# fig.savefig("../figures/" + ('Rt_AllStrain_Ex_AppxAndCircAppx_Deltat'+'_Sigma='+str(round(sigma_sel,2))+'_newPar_Rr0=2.5.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('Rt_AllStrain_Ex_AppxAndCircAppx_Deltat'+'_Sigma='+str(round(sigma_sel,2))+'_newPar_Rr0=2.5_truncatedH.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()

In [ ]:
# Plot Rt_nu_ex vs approximations (one panel per strain, rows = selected Delta t)
sigma_sel = 0.5
sigma_ind = np.argmin(np.abs(sigma_values - sigma_sel))
tem_sel = [50]

tem_list = [np.where(np.array(delta_t_values) == t)[0][0] for t in tem_sel]
nu_list = range(len(nu_values))

strain_labels = ['Variant 1 (Ancestral Virus)', 'Variant 2', 'Variant 3', 'Variant 4', 'Total']


methods = {
    "Exact": Rt_nu['ex'],
    "Zero order truncation": Rt_nu['appx_0'],
    "1st order truncation": Rt_nu['appx_1'],
    "2nd order truncation": Rt_nu['appx_2'],
    "One variants at the time": Rt_nu['appx_seq_1cir'],
    "Two variants at the time": Rt_nu['appx_seq_2cir']
}
colors = plt.cm.tab10.colors
linestyles = ["-", "--", ":", "-.", (0, (5,1)), (0, (3,1,1,1))]

colors = {
    "Exact": "#362C1E",
    "Zero order truncation": "#E9C46A",   # sand gold
    "1st order truncation": "#F09729",  
    "2nd order truncation": "#C84E13",
    # circulating family (purple scale, clearly distinct shades)
    "One variants at the time": "#B794F4" ,   # "A78BFA"soft violet
    "Two variants at the time": "#5B21B6" #"#6D28D9"    # strong violet (not too dark)
}
linestyles = [ "-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1)),]
lw=[4.5,3.5,3,2.5,2,1.5]
mrks=[9,8,7,6,5,4]
markers = {
    "Exact": "o",
    "Zero order truncation": "s",
    "1st order truncation": "^",
    "2nd order truncation": "D",
    "One variants at the time": "P",
    "Two variants at the time": "X"}

nrows = len(tem_list)
ncols = len(nu_list)
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4*ncols, 4), sharex=False, sharey=False)


if nrows == 1:
    axes = axes[np.newaxis, :]

for row, tem_ind in enumerate(tem_list):
    t_emerge = np.arange(len(nu_values)) * delta_t_values[tem_ind]
    t_emerge_indx = [int(x / dt) for x in t_emerge]
    for col, nu in enumerate(nu_list):
        ax = axes[row, col]
        ts = t_emerge_indx[nu]
        # ts=0
        # plot each method
        for k, (label, arr) in enumerate(methods.items()):
            y = arr[tem_ind, sigma_ind, nu].copy()
            # y = np.array(y, copy=False)
            # y[y == 0] = np.nan
            tend=ts+3000
            if k==4:
                if nu<3:
                    tend=ts+tem_sel[0]*10
            if k==5:
                if nu<2:
                    tend=ts+tem_sel[0]*20
            ax.plot(t_values[ts:tend], y[ts:tend],
                    color=colors[label], linestyle=linestyles[0], linewidth=lw[k], 
                    marker=markers[label],markersize=6,alpha=0.95, markevery=249, label=label)
        ax.axhline(1, color='gray', linestyle='--', linewidth=0.8)
        if row == 0:
            ax.set_title(f"{strain_labels[nu]}", fontsize=16)
        if col == 0:
            ax.set_ylabel(f"R(t)", fontsize=18)
        ax.grid(alpha=0.3)
        # ax.set_xlim([t_values[ts], t_values[ts+3000]])
        # ax.set_ylim(bottom=0)
# common labels and legend
fig.supxlabel("time (days)", fontsize=18)

handles, labels = axes[0,0].get_legend_handles_labels()
fig.legend( handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=3, fontsize=16, frameon=False,)



plt.tight_layout(rect=[0, 0, 1, 0.95])
# plt.tight_layout(rect=[0, 0.1, 1, 1])


# fig.savefig("../figures/" + ('Rt_AllStrain_Ex_AppxAndCircAppx_Deltat='+str(tem_sel[0])+'_Sigma='+str(round(sigma_sel,2))+'_newPar_R0=2.5.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('Rt_AllStrain_Ex_AppxAndCircAppx_Deltat='+str(tem_sel[0])+'_Sigma='+str(round(sigma_sel,2))+'_newPar_R0=2.5_truncatedH.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import matplotlib.patheffects as pe


sigma_sel = [0.0, 0.1,0.3,0.4, 0.5, 0.7, 0.9,1]
sigma_inds = [np.argmin(np.abs(sigma_values - s)) for s in sigma_sel]
tem_sel=[0,10,20,50,100,200]
tem_list = [np.where(np.array(delta_t_values)==t)[0][0] for t in tem_sel]
nu_list = range(4)

# sigma_colors = plt.cm.viridis(np.linspace(0,1,len(sigma_sel)))
# sigma_colors = sns.color_palette("vlag", len(sigma_sel))
# sigma_colors = [
#     "#2C7FB8", "#4C9ED9", "#7FB8E6",
#     "#B0B7C3",  # visible neutral
#     "#C2A3B8", "#C97B91", "#C74E6C", "#A61E4D" ]
sigma_colors = [
    "#1B54A3",  # blue
    "#2C7FB8",
    "#73AFDF",
    "#C2A3B8",
    "#C97B91",
    "#C74E6C",
    "#A61E4D",
    "#7F0000"]

strain_labels = ['Variant 1 (Ancestral Virus)', 'Variant 2', 'Variant 3', 'Variant 4']

nrows = len(tem_list)
ncols = 4   # 4 nus
fig, axes = plt.subplots( nrows=nrows, ncols=ncols, figsize=(5*ncols,4*nrows), sharex=False,)

# if only one row
if nrows == 1:
    axes = axes[np.newaxis, :]

for row, tem_ind in enumerate(tem_list):
    t_emerge=np.arange(0,len(nu_values))*delta_t_values[tem_ind]
    t_emerge_indx=[int(t_emerge[i]/dt) for i in range(len(t_emerge))]
    # -----------------------------------
    # columns 1-4 : individual nu
    for col, nu in enumerate(nu_list):
        ax = axes[row, col]
        eps = 1e-8
        ts=t_emerge_indx[nu]
        for color_i, (sigma, s_ind) in enumerate(zip(sigma_sel, sigma_inds)):
            y_ex=Rt_nu['ex'][tem_ind, s_ind, nu]
            line, = ax.plot(t_values[ts:ts+1500],y_ex[ts:ts+1500],color=sigma_colors[color_i],  linewidth=3,alpha=0.95, markevery=100,label=rf"$\sigma={sigma:.1f}$")

            line.set_path_effects([pe.Stroke(linewidth=5, foreground='white',alpha=0.5),pe.Normal()])

        if row == 0:
            ax.set_title(fr"{strain_labels[nu]}",fontsize=18)
        # ax.set_ylim([None, ymax*1.1])
        # ax.set_yscale('log') 
        # ax.set_ylim([10**-6, 10**-1])
        if col == 0:
            ax.set_ylabel(fr"$\Delta t = {delta_t_values[tem_ind]}$",fontsize=14)
        ax.grid(alpha=0.3)


# =========================
# COMMON LEGEND
# =========================
fig.supxlabel(r"time (days)",y=0.08,fontsize=18)
fig.supylabel(r"R(t)",fontsize=20,x=0.08)

handles, labels = axes[1,0].get_legend_handles_labels()
fig.legend( handles, labels, loc="right", ncol=1, fontsize=16, frameon=False,)
# fig.savefig("../figures/" + ('Rt_Ex_AllStrain_DeltaT_Sigma_R0=2.5.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()

In [ ]:
from matplotlib.colors import LogNorm
import matplotlib.patches as patches

app_labels = {
    "Zero order truncation ": 0,
    "1st order truncation": 1,
    "2nd order runcation": 2,
    "One variants at the time": 3,
    "Two variants at the time": 4 }

variant_labels = ['Ancestral Virus', 'Variant 1', 'Variant 2', 'Variant 3', 'Total']
fig,axes=plt.subplots(len(app_labels),len(variant_labels),figsize=(20,18),sharex=False)
axes=np.atleast_2d(axes)
plt.subplots_adjust(hspace=0.1, wspace=0.1)

delta_t_sel=[0,20,40,60,80,100,120,140,160,180,200,220,240]
delta_t_sel=delta_t_values

delta_t_index = [np.where(np.array(delta_t_values)==t)[0][0] for t in delta_t_sel]

vmin = 10**-4
vmax = (Error_r[Error_r > 0]).max()
vmax=1
norm = LogNorm(vmin=vmin, vmax=vmax)

for row,(app_name,app_ind) in enumerate(app_labels.items()):

    for var_ind,var_name in enumerate(variant_labels):
        ax=axes[row,var_ind]
        y=Error_r[app_ind,:,:,var_ind].copy()
        # y=Error_r_masked_cir[app_ind,:,:,var_ind].copy()
        y=y[delta_t_index,:]
        y[y<10**-10]=10**-10
        if app_ind >=4:
            y[0,:]=np.nan
            
        im=sns.heatmap( y,  ax=ax, 
                     xticklabels= np.round(sigma_values,2) if row == 4 else '' , yticklabels=delta_t_sel if var_ind == 0 else '', 
                     norm=norm,
                     # norm=LogNorm(),
                     cmap='magma', # cbar_kws={'label': 'Error'},
                   cbar=False)
       
        if row==0:
            ax.set_title(var_name, fontsize=16)
        if var_ind==0:
            ax.set_ylabel(app_name, fontsize=14)
            

cax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
cbar = fig.colorbar(im.collections[0],cax=cax)

# cbar = fig.colorbar(im.collections[0], ax=axes, pad=0.02,location='right')
cbar.set_label('Relative error', fontsize=14)      
    
fig.supxlabel(r"$\sigma$",fontsize=18,y=0.08)
fig.supylabel("$\Delta t$",fontsize=18,x=0.06)


# fig.savefig("../figures/" + ('RelativeError_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('RelativeError_masked_cir_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('RelativeError_masked_cir_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5_DeltaTAll.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('RelativeError_masked_cir_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5_DeltaTAll_truncatedH.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()

In [ ]:
from matplotlib.colors import LogNorm
import matplotlib.patches as patches
from matplotlib.ticker import FixedLocator

app_labels = {
    "Zero order truncation ": 0,
    "1st order truncation": 1,
    "2nd order truncation": 2,
    "One variants at the time": 3,
    "Two variants at the time": 4 } 

variant_labels = ['Variant 1 (Ancestral Virus)', 'Variant 2', 'Variant 3', 'Variant 4', 'Total']

fig,axes=plt.subplots(len(app_labels),len(variant_labels[:4]),figsize=(16,18),sharex=False)
axes=np.atleast_2d(axes)
plt.subplots_adjust(hspace=0.1, wspace=0.1)
vmin = 10**-4
vmax = (Error_r[Error_r > 0]).max()
vmax=1
norm = LogNorm(vmin=vmin, vmax=vmax)

# delta_t_sel=[0,20,40,60,80,100,120,140,160,180,200,220,240]
delta_t_sel=[0,5,10,15,20,25,30,50,100,150,200,250]
# delta_t_sel=delta_t_values

delta_t_index = [np.where(np.array(delta_t_values)==t)[0][0] for t in delta_t_sel]
sigma_ticklabels = [i/10 if i%2==0 else '' for i in range(len(sigma_values))]
delta_t_ticklabels = [i if i%40==0 else None for i in delta_t_sel]
delta_t_ticklabels = delta_t_sel
# Major ticks: 0, 40, 80, ...
major_ticks = [v for v in delta_t_sel if v % 40 == 0]
minor_ticks = [v for v in delta_t_sel if v % 20 == 0 and v % 40 != 0]

for row,(app_name,app_ind) in enumerate(app_labels.items()):
    # Error_heatmap_tmp = Error_r[app_ind] 
    # remove zeros or negatives (LogNorm requires >0)
    # vmin = (Error_heatmap_tmp[Error_heatmap_tmp > 0]).min()
    # vmin = 10**-7
    # vmax = (Error_heatmap_tmp[Error_heatmap_tmp > 0]).max()
    # norm = LogNorm(vmin=vmin, vmax=vmax)
    for var_ind,var_name in enumerate(variant_labels[:4]):
        ax=axes[row,var_ind]
        if app_ind < 3:
            y=Error_r_Var_uptopeak[app_ind,:,:,var_ind].copy()
        else:
            y=Error_r_masked_cir[app_ind,:,:,var_ind].copy()
        
        y[y<10**-10]=10**-10
        if app_ind >=3:
            y[0,:]=np.nan

        y=y[delta_t_index,:]         
        im=sns.heatmap( y,  ax=ax, 
                     xticklabels= sigma_ticklabels if row == 4 else '' , yticklabels=delta_t_ticklabels if var_ind ==0 else '', 
                     norm=norm,
                     # norm=LogNorm(),
                     cmap='magma', # cbar_kws={'label': 'Error'},
                   cbar=False)
        
        if row==0:
            ax.set_title(var_name,fontsize=16)
        if var_ind==0:
            ax.set_ylabel(app_name,fontsize=14)
            ax.set_yticklabels(delta_t_ticklabels, rotation=0)


        # fig.colorbar(im.collections[0],ax=axes[row, :],pad=0.02,location='right')
        # cbar.set_label('Error') 
            

cax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
cbar = fig.colorbar(im.collections[0],cax=cax)
cbar.set_label("Relative error",fontsize=14 )


# cbar = fig.colorbar(im.collections[0], ax=axes, pad=0.02,location='right')   
    
fig.supxlabel(r"$\sigma$",fontsize=18,y=0.08)
fig.supylabel("$\Delta t$",fontsize=18,x=0.06)


# fig.savefig("../figures/" + ('RelativeError_uptopeak_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('RelativeError_masked_cir_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('RelativeError_masked_cir_uptopeak_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5_new.png'),
#             bbox_inches='tight',dpi=300,transparent=False)

# fig.savefig("../figures/" + ('RelativeError_masked_cir_uptopeak_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5_DeltaTSelection2.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('RelativeError_masked_cir_uptopeak_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5_DeltaTSelection2_truncatedH.png'),
#             bbox_inches='tight',dpi=300,transparent=False)

# fig.savefig("../figures/" + ('RelativeError_masked_cir_uptopeak_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5_DeltaTAll.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
# fig.savefig("../figures/" + ('RelativeError_masked_cir_uptopeak_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5_DeltaTAll_truncatedH.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()

In [ ]:
import pandas as pd

approx = ["Zero order truncation", "1st order truncation", "2nd order truncation","One variants at the time","Two variants at the time" ]
delta_t_values = [ 0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 140, 150, 160, 180, 200, 220, 240, 250]
sigma_values = np.arange(0, 1.1, 0.1)
variants = ["Variant1", "Variant2", "Variant3", "Variant4"]

data = np.concatenate([
    Error_r_Var_uptopeak[[0,1,2], :, :, :],
    Error_r_masked_cir[[4,5], :, :, 1:5]
], axis=0)

A, DT, S, V = np.meshgrid(
    approx,
    delta_t_values,
    sigma_values,
    variants,
    indexing="ij"
)

df = pd.DataFrame({
    "Approximation": A.ravel(),
    "Delta_t": DT.ravel(),
    "Sigma": S.ravel(),
    "Variant": V.ravel(),
    "Error": data.ravel()
})

# df.to_csv("../data/Error_relative_uptopeak.csv", index=False)


In [ ]:
from matplotlib.colors import LogNorm
variant_labels = ["Strain1","Strain2","Strain3","Strain4","Total"]
fig,axes=plt.subplots(1,5,figsize=(25,4),sharex=True)
vmin = 10**-4
vmax=1
norm = LogNorm(vmin=vmin, vmax=vmax)
for var_ind,var_name in enumerate(variant_labels):
    y=Error_r[[0,1,2,3,4],:,:,var_ind].min(0)

    y[y<10**-6]=10**-6   
    ax=axes[var_ind]
    im=sns.heatmap( y,  ax=ax, 
                     xticklabels=np.round(sigma_values,2) , yticklabels=delta_t_values, 
                     norm=norm,
                    #  norm=LogNorm(),
                     cmap='magma', # cbar_kws={'label': 'Error'},
                   cbar=False)
        
    ax.set_title(var_name)

    
cbar = fig.colorbar(im.collections[0], ax=axes, pad=0.02,location='right')
cbar.set_label('Error')      
    
fig.supxlabel(r"$\sigma$",fontsize=14,y=-0.05,x=0.45)
fig.supylabel("$\Delta t$",fontsize=18,x=0.1)


# fig.savefig("../figures/" + ('RelativeError_heatmap_DeltaT_vs_sigma_AllStrain_BestApprx_log.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()

In [ ]:

Error_r_Var_uptopeak_mased_cir = np.load("../Data/Error_r_Var_uptopeak.npy", allow_pickle=True)
sigma_values = np.arange(0,1.1,0.1)
delta_t_values = [ 0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 140, 150, 160, 180, 200, 220, 240, 250]


In [ ]:

from matplotlib.colors import LogNorm, BoundaryNorm,ListedColormap
import matplotlib.patches as patches
from matplotlib.ticker import FixedLocator

variant_labels = ['Variant 1 (Ancestral Virus)', 'Variant 2', 'Variant 3', 'Variant 4']
fig,axes=plt.subplots(2,4,figsize=(25,10),sharex=True)
axes=np.atleast_2d(axes)
plt.subplots_adjust(hspace=0.1, wspace=0.1)
vmin = 10**-6
vmax=10
norm = LogNorm(vmin=vmin, vmax=vmax)
method_colors = [ "#4C78A8", "#F58518", "#54A24B", "#E45756", "#B279A2" ]
method_cmap = ListedColormap(method_colors)
method_norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), method_cmap.N)
delta_t_sel=[0,5,10,15,20,25,30,50,100,150,200,250]
# delta_t_sel=delta_t_values
delta_t_index = [np.where(np.array(delta_t_values)==t)[0][0] for t in delta_t_sel]
delta_t_ticklabels = delta_t_sel
sigma_ticklabels = [i/10 if i%2==0 else '' for i in range(len(sigma_values))]

for var_ind,var_name in enumerate(variant_labels):
    y=Error_r_Var_uptopeak_mased_cir[[0,1,2,3,4],:,:,var_ind].min(0)
    y=y[delta_t_index,:]    
    y[y<10**-6]=10**-6   
    ax=axes[0,var_ind]
    im_error=sns.heatmap( y,  ax=ax, 
                     xticklabels=np.round(sigma_values,2) , yticklabels=delta_t_ticklabels if var_ind ==0 else '', 
                     norm=norm,
                    #  norm=LogNorm(),
                     cmap='magma', # cbar_kws={'label': 'Error'},
                   cbar=False)
        
    ax.set_title(var_name,fontsize=20)
    if var_ind==0:
        ax.set_ylabel('The minimum error',fontsize=18)
        ax.set_yticklabels(delta_t_ticklabels, rotation=0, fontsize=14)


for var_ind,var_name in enumerate(variant_labels[:4]):
    y=np.nanargmin(Error_r_Var_uptopeak_mased_cir[[0,1,2],:,:,var_ind],axis=0)
    y=y[delta_t_index,:] 
    ax=axes[1,var_ind]
    im_method=sns.heatmap( y,  ax=ax,
                    xticklabels= sigma_ticklabels, yticklabels=delta_t_ticklabels if var_ind ==0 else '', 
                    cmap=method_cmap,
                    norm=method_norm,
                    cbar=False)
    
    ax.set_xticklabels(sigma_ticklabels, rotation=0, fontsize=14)
    if var_ind==0:
        ax.set_ylabel('Method with \nminimum error',fontsize=18)
        ax.set_yticklabels(delta_t_ticklabels, rotation=0, fontsize=14)



# This one corresponds ONLY to the error heatmaps
cax_error = fig.add_axes([
    0.91,   # x position
    0.52,   # y position
    0.015,  # width
    0.35    # height
])

cbar_error = fig.colorbar(
    im_error.collections[0],
    cax=cax_error
)

cbar_error.set_label(
    "Relative error",
    fontsize=14
)

cbar_error.set_ticks([
    1e-6,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
    1e-1,
    1,
    10
])

cbar_error.set_ticklabels([
    r"$10^{-6}$",
    r"$10^{-5}$",
    r"$10^{-4}$",
    r"$10^{-3}$",
    r"$10^{-2}$",
    r"$10^{-1}$",
    r"$10^{0}$",
    r"$>10$"
])

# COLORBAR 2: METHOD
# ============================================================

cax_method = fig.add_axes([
    0.91,   # further right
    0.12,   # lower, aligned with row 6
    0.015,
    0.35
])

cbar_method = fig.colorbar(
    im_method.collections[0],
    cax=cax_method,
    ticks=np.arange(5)
)

cbar_method.set_ticklabels([
    "Zero order \ntruncation",
    "1st order \ntruncation",
    "2nd order \ntruncation",
    "One variants \nat the time",
    "Two variants \nat the time"
])

cbar_method.set_label(
    "",
    fontsize=14
)
fig.supxlabel(r"$\sigma$",fontsize=18,y=0.05)
fig.supylabel("$\Delta t$",fontsize=18,x=0.07)
     


# fig.savefig("../figures/" + ('MinimumRelativeError_masked_cir_uptopeak_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5_Selection2.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()

In [ ]:
from matplotlib.colors import LogNorm, BoundaryNorm,ListedColormap
import matplotlib.patches as patches
from matplotlib.ticker import FixedLocator

app_labels = {
    "Zero order truncation ": 0,
    "1st order truncation": 1,
    "2nd order truncation": 2,
    "One variants at the time": 4,
    "Two variants at the time": 5 } 

variant_labels = ['Variant 1 (Ancestral Virus)', 'Variant 2', 'Variant 3', 'Variant 4', 'Total']

method_colors = [ "#4C78A8", "#F58518", "#54A24B", "#E45756", "#B279A2" ]

method_cmap = ListedColormap(method_colors)

# Boundaries between the 5 discrete categories
method_norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), method_cmap.N)

fig,axes=plt.subplots(7,len(variant_labels[:4]),figsize=(16,20),sharex=False)
axes=np.atleast_2d(axes)
plt.subplots_adjust(hspace=0.1, wspace=0.1)
vmin = 10**-4
vmax=10
norm = LogNorm(vmin=vmin, vmax=vmax)

# delta_t_sel=[0,20,40,60,80,100,120,140,160,180,200,220,240]
delta_t_sel=[0,5,10,15,20,25,30,50,100,150,200,250]
# delta_t_sel=delta_t_values

delta_t_index = [np.where(np.array(delta_t_values)==t)[0][0] for t in delta_t_sel]
sigma_ticklabels = [i/10 if i%2==0 else '' for i in range(len(sigma_values))]
delta_t_ticklabels = [i if i%40==0 else None for i in delta_t_sel]
delta_t_ticklabels = delta_t_sel
# Major ticks: 0, 40, 80, ...
major_ticks = [v for v in delta_t_sel if v % 40 == 0]
minor_ticks = [v for v in delta_t_sel if v % 20 == 0 and v % 40 != 0]

for row,(app_name,app_ind) in enumerate(app_labels.items()):
    # Error_heatmap_tmp = Error_r[app_ind] 
    # remove zeros or negatives (LogNorm requires >0)
    # vmin = (Error_heatmap_tmp[Error_heatmap_tmp > 0]).min()
    # vmin = 10**-7
    # vmax = (Error_heatmap_tmp[Error_heatmap_tmp > 0]).max()
    # norm = LogNorm(vmin=vmin, vmax=vmax)
    for var_ind,var_name in enumerate(variant_labels[:4]):
        ax=axes[row,var_ind]
        y=Error_r_Var_uptopeak_mased_cir[app_ind,:,:,var_ind].copy()

        y[y<10**-10]=10**-10
        if app_ind >=4:
            y[0,:]=np.nan

        y=y[delta_t_index,:]         
        im=sns.heatmap( y,  ax=ax, 
                     xticklabels= sigma_ticklabels if row == 5 else '' , yticklabels=delta_t_ticklabels if var_ind ==0 else '', 
                     norm=norm,
                     # norm=LogNorm(),
                     cmap='magma', # cbar_kws={'label': 'Error'},
                   cbar=False)
        
        if row==0:
            ax.set_title(var_name,fontsize=16)
        if var_ind==0:
            ax.set_ylabel(app_name,fontsize=14)
            ax.set_yticklabels(delta_t_ticklabels, rotation=0)


        # fig.colorbar(im.collections[0],ax=axes[row, :],pad=0.02,location='right')
        # cbar.set_label('Error') 

im_error = im
  
for var_ind,var_name in enumerate(variant_labels[:4]):
    y=Error_r_Var_uptopeak_mased_cir[[0,1,2,4,5],:,:,var_ind].min(0)

    y[y<10**-6]=10**-6   
    y[np.isnan(y)] = 10**-6
    y=y[delta_t_index,:] 
    ax=axes[5,var_ind]
    im=sns.heatmap( y,  ax=ax,
                    xticklabels= '', yticklabels=delta_t_ticklabels if var_ind ==0 else '', 
                    norm=norm,
                    #  norm=LogNorm(),
                    cmap='magma', # cbar_kws={'label': 'Error'},
                    cbar=False)
    if var_ind==0:
        ax.set_ylabel('The minimum error',fontsize=14)
        ax.set_yticklabels(delta_t_ticklabels, rotation=0)

for var_ind,var_name in enumerate(variant_labels[:4]):
    y=Error_r_Var_uptopeak_mased_cir[[0,1,2,4,5],:,:,var_ind].argmin(0)
    y=y[delta_t_index,:] 
    ax=axes[6,var_ind]
    im_method=sns.heatmap( y,  ax=ax,
                    xticklabels= sigma_ticklabels, yticklabels=delta_t_ticklabels if var_ind ==0 else '', 
                    cmap=method_cmap,
                    norm=method_norm,
                    cbar=False)
    if var_ind==0:
        ax.set_ylabel('Method with \nminimum error',fontsize=14)
        ax.set_yticklabels(delta_t_ticklabels, rotation=0)



# This one corresponds ONLY to the error heatmaps
cax_error = fig.add_axes([
    0.92,   # x position
    0.40,   # y position
    0.015,  # width
    0.40    # height
])

cbar_error = fig.colorbar(
    im_error.collections[0],
    cax=cax_error
)

cbar_error.set_label(
    "Relative error",
    fontsize=14
)

cbar_error.set_ticks([
    1e-4,
    1e-3,
    1e-2,
    1e-1,
    1,
    10
])

cbar_error.set_ticklabels([
    r"$10^{-4}$",
    r"$10^{-3}$",
    r"$10^{-2}$",
    r"$10^{-1}$",
    r"$10^{0}$",
    r"$>10$"
])

# COLORBAR 2: METHOD
# ============================================================

cax_method = fig.add_axes([
    0.92,   # further right
    0.11,   # lower, aligned with row 6
    0.015,
    0.1
])

cbar_method = fig.colorbar(
    im_method.collections[0],
    cax=cax_method,
    ticks=np.arange(5)
)

cbar_method.set_ticklabels([
    "Zero order \ntruncation",
    "1st order \ntruncation",
    "2nd order \ntruncation",
    "One variants \nat the time",
    "Two variants \nat the time"
])

cbar_method.set_label(
    "",
    fontsize=14
)



# cax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
# cbar = fig.colorbar(im.collections[0],cax=cax, )
# cbar.set_label("Relative error",fontsize=14 )
# cbar.set_ticks([1e-4, 1e-3, 1e-2, 1e-1, 1, 10])
# cbar.set_ticklabels([
#     r"$10^{-4}$",
#     r"$10^{-3}$",
#     r"$10^{-2}$",
#     r"$10^{-1}$",
#     r"$10^{0}$",
#     r"$>10$"
# ])

# cbar = fig.colorbar(im.collections[0], ax=axes, pad=0.02,location='right')   
    
fig.supxlabel(r"$\sigma$",fontsize=18,y=0.08)
fig.supylabel("$\Delta t$",fontsize=18,x=0.06)



# fig.savefig("../figures/" + ('RelativeError_masked_cir_uptopeak_heatmap_DeltaT_vs_sigma_AllStrain_AppxAndCircAppx_log_R0=2.5_Selection2_MinError.png'),
#             bbox_inches='tight',dpi=300,transparent=False)
plt.show()

In [ ]:
from matplotlib.colors import TwoSlopeNorm

Rave_nu_ex = np.zeros((len(nu_list),len(delta_t_values),len(sigma_values)))

for dt_ind, deltaT in enumerate(delta_t_values):
    t_emerge_indx = (np.arange(len(nu_values)) * deltaT / dt).astype(int)
    for var_pos, var_ind in enumerate(nu_list):
        ts = t_emerge_indx[var_ind]
        Rave_nu_ex[var_pos, dt_ind] = (Rt_nu_ex[dt_ind, :, var_ind, ts:ts+10].mean(axis=1) )
    
fig,axes=plt.subplots(1,4,figsize=(20,4),sharex=True)
norm = TwoSlopeNorm( vmin=np.nanmin(Rave_nu_ex), vcenter=1.0, vmax=np.nanmax(Rave_nu_ex))

for var_ind,var_name in enumerate(nu_list):
    y=Rave_nu_ex[var_ind,:,:]
    ax=axes[var_ind]
    im=sns.heatmap( y,  ax=ax, 
                     xticklabels=np.round(sigma_values,2) , yticklabels=delta_t_values,center=1,
                     # norm=norm,
                     # norm=LogNorm(),
                     cmap='RdBu_r', # cbar_kws={'label': 'Error'},
                   cbar=False)
    ax.set_title(var_name)
    

cbar = fig.colorbar(im.collections[0], ax=axes, pad=0.02,location='right')
cbar.set_label('R_ave')      
    
fig.supxlabel(r"$\sigma$",fontsize=14,y=-0.05,x=0.45)
fig.supylabel("$\Delta t$",fontsize=18,x=0.1)

plt.show()

In [ ]:
overlapTime=[]
for tem_ind in range(Inc_ex.shape[0]):
    overlapTime_tmp=[]
    for s_ind in range(Inc_ex.shape[1]):
        y_ex=Inc_ex[tem_ind, s_ind, :].sum(1)
        y_ex[y_ex<10**-7]=0
        y_ex[y_ex>0]=1
        ovtime = np.diag(np.matmul(y_ex,np.transpose(y_ex.sum(0)-y_ex)))*dt
        overlapTime_tmp.append(ovtime)
    overlapTime.append(overlapTime_tmp)

overlapTime = np.array(overlapTime)

In [ ]:
overlapTime
fig,axes=plt.subplots(1,4,figsize=(20,4),sharex=True)

for var_ind,var_name in enumerate(variant_labels[:4]):
    y=overlapTime[:,:,var_ind]
    ax=axes[var_ind]
    im=sns.heatmap( y,  ax=ax, 
                     xticklabels=np.round(sigma_values,2) , yticklabels=delta_t_values, 
                     # norm=norm,
                     # norm=LogNorm(),
                     cmap='magma', # cbar_kws={'label': 'Error'},
                   cbar=False)
        
    ax.set_title(var_name)

    
cbar = fig.colorbar(im.collections[0], ax=ax, pad=0.02,location='right')
cbar.set_label('overlaped time')      
    
fig.supxlabel(r"$\sigma$",fontsize=14)
fig.supylabel("$\Delta t$",fontsize=18,x=0.1)